# EDA and Data Preprocessing
**Dataset:** Telecom Market in Bangladesh (GP Model)

This notebook loads the raw CSV, reshapes it into a tidy long format, and prepares it for analysis.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 20)

## 2. Load Raw Data
The source file is a wide-format spreadsheet (months as columns). We load it without a header and inspect.

In [2]:
raw = pd.read_csv('GP Model.xlsx - Telecom Mkt. in Bangladesh.csv', header=None)
print('Raw shape:', raw.shape)
raw.head(12)

Raw shape: (78, 112)


,0,1,2,3,4,5,6,7,8,9,...,102,103,104,105,106,107,108,109,110,111
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,...,Average (12 Months),Assumption,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Grameen,16.88,17.2,17.81,18.6,19.58,20.31,20.84,20.84,...,0.6%,0.65%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Banglalink,7.45,7.88,8.31,8.64,8.99,9.46,9.90,10.17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Robi,7.28,7.36,7.45,7.57,7.71,7.85,7.98,8.14,...,45%,40.0%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Airtel,2.37,2.60,2.79,2.93,3.13,3.31,3.48,3.68,...,20%,30.0%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Citycell,1.45,1.51,1.56,1.60,1.64,1.70,1.67,1.68,...,24%,22.0%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,Teletalk,1.00,1.00,1.01,0.99,0.99,1.07,0.93,0.89,...,10%,4.0%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Extract the Subscriber Block
Rows 4–10 contain monthly subscriber counts (in millions) for each operator from Jan-2008 to Dec-2015 (96 months).

In [3]:
years = list(range(2008, 2016))
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
date_cols = [f'{m}-{y}' for y in years for m in months]

operators = ['Grameenphone', 'Banglalink', 'Robi', 'Airtel', 'Citycell', 'Teletalk']
subs = raw.iloc[4:10, 2:2+96].copy()
subs.columns = date_cols
subs.index = operators
subs = subs.apply(pd.to_numeric, errors='coerce')
subs.head()

,Jan-2008,Feb-2008,Mar-2008,Apr-2008,May-2008,Jun-2008,Jul-2008,Aug-2008,Sep-2008,Oct-2008,...,Mar-2015,Apr-2015,May-2015,Jun-2015,Jul-2015,Aug-2015,Sep-2015,Oct-2015,Nov-2015,Dec-2015
Grameenphone,16.88,17.20,17.81,18.60,19.58,20.31,20.84,20.84,20.82,20.79,...,52.01,52.35,52.84,53.13,53.98,55.04,55.51,55.85,56.20,56.54
Banglalink,7.45,7.88,8.31,8.64,8.99,9.46,9.90,10.17,10.14,10.17,...,31.92,32.04,32.10,32.22,32.41,32.88,32.62,32.87,33.13,33.39
Robi,7.28,7.36,7.45,7.57,7.71,7.85,7.98,8.14,7.63,7.64,...,26.29,26.63,27.14,27.37,27.92,28.32,28.37,28.56,28.75,28.94
Airtel,2.37,2.60,2.79,2.93,3.13,3.31,3.48,3.68,3.86,2.25,...,8.19,8.35,8.52,8.74,9.08,9.39,9.72,9.75,9.79,9.82
Citycell,1.45,1.51,1.56,1.60,1.64,1.70,1.67,1.68,1.74,1.77,...,1.25,1.23,1.21,1.19,1.16,1.14,1.11,1.09,1.07,1.05


## 4. Reshape to Long (Tidy) Format
We melt the wide table so each row represents one (Operator, Month) observation.

In [4]:
df = subs.reset_index().rename(columns={'index': 'Operator'})
df = df.melt(id_vars='Operator', var_name='Period', value_name='Subscribers_Mn')
df['Date'] = pd.to_datetime(df['Period'], format='%b-%Y')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month_name()
df = df[['Operator', 'Date', 'Year', 'Month', 'Subscribers_Mn']]
df = df.sort_values(['Operator', 'Date']).reset_index(drop=True)
df.head()

,Operator,Date,Year,Month,Subscribers_Mn
0,Airtel,2008-01-01,2008,January,2.37
1,Airtel,2008-02-01,2008,February,2.60
2,Airtel,2008-03-01,2008,March,2.79
3,Airtel,2008-04-01,2008,April,2.93
4,Airtel,2008-05-01,2008,May,3.13


In [5]:
print('Tidy shape:', df.shape)
df.info()

Tidy shape: (576, 5)
<class 'pandas.DataFrame'>
RangeIndex: 576 entries, 0 to 575
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Operator        576 non-null    str           
 1   Date            576 non-null    datetime64[us]
 2   Year            576 non-null    int32         
 3   Month           576 non-null    str           
 4   Subscribers_Mn  576 non-null    float64       
dtypes: datetime64[us](1), float64(1), int32(1), str(2)
memory usage: 28.5 KB


## 5. Check Missing Values
**Why:** Missing values can break aggregations and distort summary statistics.

In [6]:
print('Missing per column:')
print(df.isnull().sum())
print('\nRows with missing values:', df.isnull().any(axis=1).sum())

Missing per column:
Operator          0
Date              0
Year              0
Month             0
Subscribers_Mn    0
dtype: int64

Rows with missing values: 0


In [7]:
df['Subscribers_Mn'] = df.groupby('Operator')['Subscribers_Mn'].transform(
    lambda s: s.interpolate().ffill().bfill()
)
print('Missing after interpolation:', df.isnull().sum().sum())

Missing after interpolation: 0


## 6. Check Duplicate Records
**Why:** Duplicates inflate counts and bias statistics.

In [8]:
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate (Operator, Date) keys:', df.duplicated(subset=['Operator','Date']).sum())

Duplicate rows: 0
Duplicate (Operator, Date) keys: 0


## 7. Feature Engineering
Add Month-over-Month growth and market share — these enrich the analysis.

In [9]:
df['MoM_Growth_%'] = df.groupby('Operator')['Subscribers_Mn'].pct_change() * 100

sector_total = df.groupby('Date')['Subscribers_Mn'].transform('sum')
df['Market_Share_%'] = (df['Subscribers_Mn'] / sector_total) * 100
df.head()

,Operator,Date,Year,Month,Subscribers_Mn,MoM_Growth_%,Market_Share_%
0,Airtel,2008-01-01,2008,January,2.37,NaN,6.505627
1,Airtel,2008-02-01,2008,February,2.60,9.704641,6.924101
2,Airtel,2008-03-01,2008,March,2.79,7.307692,7.166709
3,Airtel,2008-04-01,2008,April,2.93,5.017921,7.265063
4,Airtel,2008-05-01,2008,May,3.13,6.825939,7.445290


## 8. Statistical Summary

In [10]:
df.describe()

,Date,Year,Subscribers_Mn,MoM_Growth_%,Market_Share_%
count,576,576.000000,576.000000,570.000000,576.000000
mean,2011-12-16 11:00:00,2011.500000,14.026972,1.234583,16.666667
min,2008-01-01 00:00:00,2008.000000,0.840000,-41.709845,0.783465
25%,2009-12-24 06:00:00,2009.750000,1.907500,0.144262,3.124695
50%,2011-12-16 12:00:00,2011.500000,8.335000,1.059886,12.741184
75%,2013-12-08 18:00:00,2013.250000,23.960000,2.288197,25.653084
max,2015-12-01 00:00:00,2015.000000,56.540000,20.437956,47.804093
std,NaN,2.293279,14.108073,3.360210,14.889473


In [11]:
df.groupby('Operator')['Subscribers_Mn'].agg(['min','mean','max','std']).round(2)

,min,mean,max,std
Operator,,,,
Airtel,2.20,5.69,9.82,2.43
Banglalink,7.45,21.68,33.39,8.08
Citycell,1.05,1.61,2.00,0.27
Grameenphone,16.88,36.03,56.54,12.16
Robi,7.28,17.22,28.94,7.20
Teletalk,0.84,1.93,4.28,1.18


## 9. Outlier Detection (IQR method)

In [12]:
q1 = df['MoM_Growth_%'].quantile(0.25)
q3 = df['MoM_Growth_%'].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = df[(df['MoM_Growth_%'] < lower) | (df['MoM_Growth_%'] > upper)]
print(f'IQR bounds: [{lower:.2f}, {upper:.2f}]')
print(f'Outlier rows: {len(outliers)}')
outliers.head()

IQR bounds: [-3.07, 5.50]
Outlier rows: 51


,Operator,Date,Year,Month,Subscribers_Mn,MoM_Growth_%,Market_Share_%
1,Airtel,2008-02-01,2008,February,2.60,9.704641,6.924101
2,Airtel,2008-03-01,2008,March,2.79,7.307692,7.166709
4,Airtel,2008-05-01,2008,May,3.13,6.825939,7.445290
5,Airtel,2008-06-01,2008,June,3.31,5.750799,7.574371
7,Airtel,2008-08-01,2008,August,3.68,5.747126,8.105727


## 10. Save Cleaned Dataset

In [13]:
df.to_csv('telecom_clean.csv', index=False)
print('Saved telecom_clean.csv with shape', df.shape)

Saved telecom_clean.csv with shape (576, 7)
